# Analysis: Predicting Intraprostatic Recurrence on MRI After Radiotherapy

Primary analysis notebook — descriptive statistics, stratification, and visualizations.

All logic lives in `pca_mri.analysis` and `pca_mri.visualization`.  
Load the cleaned dataset produced by `data-cleanup.ipynb` before running.

## Sections
1. Flow diagram of patient selection and cohort derivation
2. Descriptive statistics tables

In [7]:
import sys
sys.path.insert(0, '.')  # make pca_mri importable from repo root

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('dark_background')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

from pca_mri import io
from pca_mri.analysis import descriptive, stratification
from pca_mri.visualization import distributions, timelines, kinetics
from pca_mri.visualization.sankey import plot_exclusion_sankey
import glob

In [8]:
# Automatically pick the most recent cleaned file
csv_files = sorted(glob.glob('df-clean-v1_*.csv'))
clean_path = csv_files[-1]
print(f'Loading: {clean_path}')

df = io.load_clean(clean_path)
print(f'Shape: {df.shape}')

Loading: df-clean-v1_20260318_071524.csv
Shape: (68, 63)


## 1. Flow diagram of patient selection and cohort derivation

In [9]:
fig_sankey = plot_exclusion_sankey()
fig_sankey.show()

## 2. Descriptive statistics tables
Continuous: Median [Q1–Q3], Kruskal-Wallis p-value.  
Categorical: N (%), chi-squared p-value.
* a) By treatment type
* b) By MRI recurrence detection (positive, negative)
* c) By biopsy recurrence (positive, negative)
* d) By CAPRA risk group (low, intermediate, high)

In [10]:
t2 = df.describe()
t2_a = descriptive.table1(df, stratify_by='tx-type', title = "Descriptive Statistics Stratified by Treatment")
t2_b = descriptive.table1(df, stratify_by='rec_mri-result', title = "Descriptive Statistics Stratified by MRI Recurrence Result")
t2_c = descriptive.table1(df, stratify_by='biopsy-result', title = "Descriptive Statistics Stratified by Biopsy Result")
t2_d = descriptive.table1(df, stratify_by='capra-risk_group', title = "Descriptive Statistics Stratified by CAPRA Risk Group")

## 3. Missing data report

In [11]:
t3 = descriptive.missing_data_summary(df)
t3_styled = descriptive.style_missing_table(t3)

In [12]:
t3_styled

Variable,Number missing,Number present,% missing
PSA doubling time to recurrence MRI (months),36,32,52.9%
Time to MRI recurrence (days),16,52,23.5%
Time to biochemical failure (days),13,55,19.1%
PSA difference to recurrence MRI,2,66,2.9%
Age (years),0,68,0.0%
CAPRA risk group,0,68,0.0%
PSA nadir < 0.2 ng/mL,0,68,0.0%
Androgen deprivation therapy (ADT),0,68,0.0%
Clinical T-stage,0,68,0.0%
Gleason score at diagnosis,0,68,0.0%


## 3. Descriptive statistics visualization

In [13]:
import ipywidgets as widgets
from IPython.display import display
import plotly.graph_objects as go
import plotly.colors as pc
import numpy as np
from scipy.stats import gaussian_kde
from pca_mri.analysis.descriptive import _CATEGORICAL, _CONTINUOUS

# Build dropdown options from _CATEGORICAL / _CONTINUOUS (label shown, column value used)
# Filter to columns present in the loaded df
cat_options  = [(label, col) for col, label in _CATEGORICAL if col in df.columns]
cont_options = [(label, col) for col, label in _CONTINUOUS  if col in df.columns]

cat_dd = widgets.Dropdown(
    options=cat_options,
    value='tx-type',
    description='Categorical:',
    style={'description_width': 'initial'},
)
cont_dd = widgets.Dropdown(
    options=cont_options,
    value='tx-age',
    description='Continuous:',
    style={'description_width': 'initial'},
)

COLOR_SEQ = pc.qualitative.Plotly
out = widgets.Output()


def update(cat, cont):
    out.clear_output(wait=True)
    with out:
        cats   = df[cat].dropna().unique()
        colors = {c: COLOR_SEQ[i % len(COLOR_SEQ)] for i, c in enumerate(cats)}

        # Lookup human-readable labels
        cont_label = next((label for col, label in _CONTINUOUS  if col == cont), cont)
        cat_label  = next((label for col, label in _CATEGORICAL if col == cat),  cat)

        # ── Plot 1: PDF (KDE) of the continuous variable, one curve per category ──
        fig1 = go.Figure()
        for cat_val in cats:
            vals = (
                df.loc[df[cat] == cat_val, cont]
                .pipe(pd.to_numeric, errors='coerce')
                .replace([np.inf, -np.inf], np.nan)
                .dropna()
            )
            if len(vals) < 2:
                continue
            kde    = gaussian_kde(vals, bw_method='scott')
            span   = vals.max() - vals.min() or 1
            x_grid = np.linspace(vals.min() - 0.1 * span, vals.max() + 0.1 * span, 300)
            fig1.add_trace(go.Scatter(
                x=x_grid, y=kde(x_grid),
                mode='lines',
                name=str(cat_val),
                line=dict(color=colors[cat_val], width=2),
                hovertemplate=(
                    f'<b>{cat_val}</b><br>'
                    f'{cont_label}: %{{x:.2f}}<br>'
                    f'Density: %{{y:.4f}}<extra></extra>'
                ),
            ))
        fig1.update_layout(
            title=f'PDF of <b>{cont_label}</b> stratified by <b>{cat_label}</b>',
            xaxis_title=cont_label,
            yaxis_title='Probability density',
            legend_title=cat_label,
            template='plotly_dark',
            height=420,
        )
        fig1.show()

        # ── Plot 2: stacked bar — one segment per category group ──
        total = int(df[cat].notna().sum())
        fig2  = go.Figure()
        for cat_val in cats:
            count = int((df[cat] == cat_val).sum())
            pct   = count / total * 100 if total else 0
            fig2.add_trace(go.Bar(
                name=str(cat_val),
                x=[cat_label],
                y=[count],
                marker_color=colors[cat_val],
                text=f'{cat_val}<br>N={count} ({pct:.1f}%)',
                textposition='inside',
                hovertemplate=(
                    f'<b>{cat_val}</b><br>'
                    f'N = {count}<br>'
                    f'{pct:.1f}% of {total}<extra></extra>'
                ),
            ))
        fig2.update_layout(
            barmode='stack',
            title=f'Distribution of <b>{cat_label}</b>',
            yaxis_title='Count',
            legend_title=cat_label,
            template='plotly_dark',
            height=380,
        )
        fig2.show()


cat_dd.observe( lambda _: update(cat_dd.value, cont_dd.value), names='value')
cont_dd.observe(lambda _: update(cat_dd.value, cont_dd.value), names='value')

display(widgets.VBox([widgets.HBox([cat_dd, cont_dd]), out]))
update(cat_dd.value, cont_dd.value)

## 5. Stratification: LDR / HDR / RT

In [ ]:
# CORRECTION: The dataset includes patients with positive biopsy (confirmed recurrence),
# negative biopsy, and patients without a biopsy result — NOT all patients are recurrent.
# We now report biopsy status per MRI result group.

_POSITIVE_BIOPSY = {'positif', 'positive', 'positiv'}
_NEGATIVE_BIOPSY = {'négative', 'négatif', 'negative', 'negativ'}
_MRI_POSITIVE   = {'positive', 'positif', 'positiv'}
_MRI_NEGATIVE   = {'négative', 'negative', 'négatif', 'negativ'}
_MRI_EQUIVOCAL  = {'equivoque'}

def _biopsy_norm(series):
    return series.astype(str).str.strip().str.lower()

# ── Build MRI result groups ──
_mri_norm = df['rec_mri-result'].astype(str).str.strip().str.lower()
mri_groups = {
    'MRI+':      df[_mri_norm.isin(_MRI_POSITIVE)],
    'MRI−':      df[_mri_norm.isin(_MRI_NEGATIVE)],
    'Equivocal': df[_mri_norm.isin(_MRI_EQUIVOCAL)],
    'Unknown':   df[~_mri_norm.isin(_MRI_POSITIVE | _MRI_NEGATIVE | _MRI_EQUIVOCAL)],
}
mri_groups = {k: v for k, v in mri_groups.items() if len(v) > 0}

# ── Biopsy result breakdown per MRI result ──
print('Biopsy result by MRI result:')
print(f'{"Group":<12}  {"N":>4}  {"Biopsy+":>9}  {"Biopsy-":>9}  {"No biopsy":>10}')
for name, subset in mri_groups.items():
    norm      = _biopsy_norm(subset['biopsy-result'])
    n_pos     = norm.isin(_POSITIVE_BIOPSY).sum()
    n_neg     = norm.isin(_NEGATIVE_BIOPSY).sum()
    n_missing = subset['biopsy-result'].isna().sum()
    print(f'{name:<12}  {len(subset):>4}  {n_pos:>9}  {n_neg:>9}  {n_missing:>10}')

print()

# ── Time to MRI recurrence — biopsy-positive patients only ──
# (Restricted because recurrence time is only clinically meaningful for confirmed cases)
print('Time to MRI recurrence (biopsy-positive patients only):')
for name, subset in mri_groups.items():
    norm       = _biopsy_norm(subset['biopsy-result'])
    subset_pos = subset[norm.isin(_POSITIVE_BIOPSY)]
    valid      = pd.to_numeric(subset_pos['rec_mri-time_to_rec-days'], errors='coerce').dropna()
    if len(valid) >= 1:
        print(f'  {name}: N={len(subset_pos)}, median = {valid.median():.0f} days '
              f'[{valid.quantile(0.25):.0f}–{valid.quantile(0.75):.0f}]')
    else:
        print(f'  {name}: N={len(subset_pos)}, insufficient data')

## 6. Stratification: biopsy result and MRI detection

The cohort is split by **biopsy result** (primary definition of confirmed recurrence) and by **MRI detection status**.

- **Biopsy-positive**: recurrence histologically confirmed (N ≈ 46)
- **Biopsy-negative**: biopsy performed but no confirmed recurrence (N ≈ 20)
- **No biopsy**: 41 patients have no biopsy result — excluded from biopsy-based analyses

A cross-tabulation of biopsy result vs. MRI detection is provided to characterise agreement between imaging and pathology.

In [ ]:
# CORRECTION: Previous version assumed all patients were biopsy-confirmed recurrent.
# Biopsy result is now treated as a variable, not an assumption.

# ── Split by biopsy result (excludes patients with no biopsy) ──
biopsy_pos, biopsy_neg = stratification.split_by_recurrence(df, method='biopsy')
n_no_biopsy = df['biopsy-result'].isna().sum()

print(f'Biopsy-positive (confirmed recurrence):          N={len(biopsy_pos)}')
print(f'Biopsy-negative (no confirmed recurrence):       N={len(biopsy_neg)}')
print(f'No biopsy result (excluded from biopsy analyses): N={n_no_biopsy}')
print()

# ── Split by MRI detection status (whole cohort) ──
rec_mri, non_rec_mri = stratification.split_by_recurrence(df, method='mri')
print(f'MRI-detected recurrence (whole cohort):          N={len(rec_mri)}')
print(f'No positive MRI:                                 N={len(non_rec_mri)}')
print()

# ── Cross-tabulation: biopsy result vs MRI detection ──
# Patients with no biopsy are excluded; they do not contribute to either biopsy group.
_norm = df['biopsy-result'].astype(str).str.strip().str.lower()
has_biopsy = _norm.isin(_POSITIVE_BIOPSY | _NEGATIVE_BIOPSY)
df_biopsied = df[has_biopsy].copy()
df_biopsied['biopsy_status'] = _norm[has_biopsy].apply(
    lambda x: 'Positive' if x in _POSITIVE_BIOPSY else 'Negative'
)
df_biopsied['mri_positive'] = df_biopsied['rec_mri-index'].notna().map(
    {True: 'MRI+', False: 'MRI−'}
)

crosstab = pd.crosstab(
    df_biopsied['biopsy_status'],
    df_biopsied['mri_positive'],
    rownames=['Biopsy result'],
    colnames=['MRI detection'],
    margins=True,
)
print('Cross-tabulation: biopsy result vs MRI-detected recurrence')
print(crosstab)

## 7. Distributions

In [ ]:
fig = distributions.plot_age_distribution(df)
plt.show()

In [ ]:
fig = distributions.plot_psa_distribution(df)
plt.show()

In [ ]:
fig = distributions.plot_gleason_distribution(df)
plt.show()

In [ ]:
fig = distributions.plot_t_stage_distribution(df)
plt.show()

In [ ]:
fig = distributions.plot_capra_distribution(df)
plt.show()

## 8. Time-to-event timelines

In [ ]:
fig = timelines.plot_time_to_bf(df)
plt.show()

In [ ]:
# Time to MRI recurrence — whole cohort (includes biopsy-negative and unbiopsy'd patients)
fig = timelines.plot_time_to_rec_mri(df)
plt.title('Time to MRI recurrence — whole cohort')
plt.show()

# CORRECTION: Also plot restricted to biopsy-positive patients, where recurrence is confirmed.
fig = timelines.plot_time_to_rec_mri(biopsy_pos)
plt.title('Time to MRI recurrence — biopsy-positive patients only (confirmed recurrence)')
plt.show()

In [ ]:
# BF-to-MRI lag — biopsy-positive patients only
# (The lag between biochemical failure and MRI recurrence detection is only
#  clinically meaningful when recurrence is histologically confirmed.)
fig = timelines.plot_bf_to_mri_lag(biopsy_pos)
plt.title('BF-to-MRI recurrence lag — biopsy-positive patients only')
plt.show()

In [ ]:
fig = timelines.plot_mri_followup_count(df)
plt.show()

## 9. PSA kinetics

In [ ]:
fig = kinetics.plot_psa_doubling_time(df)
plt.show()

In [ ]:
# PSA trajectories — whole cohort
fig = kinetics.plot_psa_trajectory(df)
plt.title('PSA trajectories — whole cohort')
plt.show()

# CORRECTION: Stratify PSA trajectories by biopsy result.
# Previously the analysis assumed all patients were recurrent; we now compare
# biopsy-positive vs biopsy-negative patients to assess whether PSA kinetics
# differ between confirmed recurrence and non-recurrence.
fig = kinetics.plot_psa_trajectory(biopsy_pos)
plt.title('PSA trajectories — biopsy-positive (confirmed recurrence, N={})'.format(len(biopsy_pos)))
plt.show()

fig = kinetics.plot_psa_trajectory(biopsy_neg)
plt.title('PSA trajectories — biopsy-negative (no confirmed recurrence, N={})'.format(len(biopsy_neg)))
plt.show()

## 10. Sensitivity analyses

Two sensitivity analyses are performed:

**a) Excluding converter patients** — patients whose MRI changed from positive to negative over follow-up (`is_converter`).

**b) Biopsy-positive patients only** — restricts the cohort to histologically confirmed recurrence cases, which is the clinically most relevant subgroup for recurrence prediction modelling.

In [ ]:
### a) Excluding converter patients
df_no_conv = df[~df['is_converter']].copy()
print(f'Converters excluded: {df["is_converter"].sum()}')
print(f'Remaining N: {len(df_no_conv)}')

In [ ]:
# Table 1 excluding converters
t1_no_conv = descriptive.table1(df_no_conv, stratify_by='tx-type')
descriptive.style_table(t1_no_conv)

In [ ]:
# PSA trajectory excluding converters
fig = kinetics.plot_psa_trajectory(df_no_conv)
plt.title('PSA trajectories (converters excluded)')
plt.show()

### b) Biopsy-positive patients only (confirmed recurrence)

This subgroup (N ≈ 46) is the primary target population for recurrence prediction modelling.  
All analyses below are restricted to patients with histologically confirmed intraprostatic recurrence.

In [ ]:
# CORRECTION: biopsy_pos was defined in Step 6 using split_by_recurrence(df, method='biopsy').
# This is the correct subgroup for recurrence prediction — patients with confirmed recurrence.

print(f'Biopsy-positive patients: N={len(biopsy_pos)}')
print(f'  of whom, converters:    N={biopsy_pos["is_converter"].sum()}')

# Table 1 for biopsy-positive patients, stratified by treatment type
t1_biopsy_pos = descriptive.table1(
    biopsy_pos,
    stratify_by='tx-type',
    title='Clinical characteristics — biopsy-positive patients by treatment type',
)
descriptive.style_table(t1_biopsy_pos)

In [ ]:
# PSA trajectory for biopsy-positive patients (confirmed recurrence)
fig = kinetics.plot_psa_trajectory(biopsy_pos)
plt.title(f'PSA trajectories — biopsy-positive patients (N={len(biopsy_pos)})')
plt.show()

# PSA doubling time for biopsy-positive patients
fig = kinetics.plot_psa_doubling_time(biopsy_pos)
plt.title(f'PSA doubling time — biopsy-positive patients (N={len(biopsy_pos)})')
plt.show()

## 11. Statistical Analysis — Full Pipeline & Dashboard

Implements the statistical analysis plan (SAP):
1. **Objective 1:** Prevalence of MRI-visible recurrence with 95% CI
2. **Objective 2:** Diagnostic accuracy of MRI (vs biopsy reference standard)
3. **Objective 3a:** Univariate logistic regression (16 candidate predictors)
4. **Objective 3b:** Multivariable logistic regression with VIF check & bootstrap validation
5. **Interactive dashboard** combining all results

In [ ]:
from pca_mri.analysis import diagnostic, regression
from pca_mri.visualization.dashboard import show_dashboard

### 11.1 Objective 1 — Prevalence of MRI-Positive Recurrence
* Method to compute p-value: Chi-square test.
* Method to 95% confidence intervals: Clopper–Pearson method.

In [ ]:
# Overall prevalence with 95% Clopper-Pearson CI
prev = diagnostic.prevalence(df)
print(f"MRI-Positive Recurrence Prevalence")
print(f"  {prev['n_positive']}/{prev['n_total']} = {prev['prevalence']:.1%}")
print(f"  95% CI: [{prev['ci_lower']:.1%} – {prev['ci_upper']:.1%}]")
print()

# Prevalence by treatment type
prev_tx = diagnostic.prevalence_by_subgroup(df, by="tx-type")
print("Prevalence by treatment type:")
for _, row in prev_tx.iterrows():
    print(f"  {row['group']}: {row['n_positive']}/{row['n_total']} = {row['prevalence']:.1%} "
          f"[{row['ci_lower']:.1%} – {row['ci_upper']:.1%}]")
if "p_value" in prev_tx.attrs:
    print(f"  p = {prev_tx.attrs['p_value']:.3f}")
print()

# Prevalence by CAPRA risk group
prev_capra = diagnostic.prevalence_by_subgroup(df, by="capra-risk_group")
print("Prevalence by CAPRA risk group:")
for _, row in prev_capra.iterrows():
    print(f"  {row['group']}: {row['n_positive']}/{row['n_total']} = {row['prevalence']:.1%} "
          f"[{row['ci_lower']:.1%} – {row['ci_upper']:.1%}]")
if "p_value" in prev_capra.attrs:
    print(f"  p = {prev_capra.attrs['p_value']:.3f}")

### 11.2 Objective 2 — Diagnostic Accuracy of MRI
* Method to 95% confidence intervals: Clopper–Pearson method.

In [ ]:
# 2x2 contingency table: MRI vs biopsy
ct = diagnostic.contingency_table(df)
print("2x2 Contingency Table (MRI vs Biopsy):")
print(ct)
print()

# Diagnostic accuracy metrics with 95% CI
acc = diagnostic.diagnostic_accuracy(df)
print("Diagnostic Accuracy Metrics (biopsy = reference standard):")
for _, row in acc.iterrows():
    if np.isnan(row['ci_lower']):
        print(f"  {row['metric']}: {row['value']:.3f}")
    else:
        print(f"  {row['metric']}: {row['value']:.3f} [95% CI: {row['ci_lower']:.3f} – {row['ci_upper']:.3f}]")

### 11.3 Objective 3a — Univariate Logistic Regression

In [ ]:
# Univariate logistic regression: each predictor vs MRI+ outcome
uni = regression.univariate_screen(df)
print(f"Univariate logistic regression results ({len(uni)} predictor terms):\n")

display_cols = ['label', 'param', 'or', 'ci_lower', 'ci_upper', 'p_value', 'p_adj', 'n']
uni_display = uni[display_cols].copy()
uni_display.columns = ['Variable', 'Parameter', 'OR', '95% CI Lower', '95% CI Upper', 'p-value', 'p-adj (FDR)', 'N']

# Format for display
uni_display['OR'] = uni_display['OR'].map(lambda x: f"{x:.3f}")
uni_display['95% CI Lower'] = uni_display['95% CI Lower'].map(lambda x: f"{x:.3f}")
uni_display['95% CI Upper'] = uni_display['95% CI Upper'].map(lambda x: f"{x:.3f}")
uni_display['p-value'] = uni_display['p-value'].map(lambda x: f"{x:.3f}" if x >= 0.001 else "<0.001")
uni_display['p-adj (FDR)'] = uni_display['p-adj (FDR)'].map(lambda x: f"{x:.3f}" if x >= 0.001 else "<0.001")

uni_display

### 11.4 Objective 3b — Multivariable Logistic Regression

Clinically-guided model using CAPRA components + time to biochemical failure. VIF > 5 triggers automatic removal of collinear predictors.

In [ ]:
# Multivariable model — automatic predictor selection
mv = regression.build_multivariable_model(df)

print(f"Multivariable Logistic Regression (N={mv['n']}, events={mv['n_events']})")
print(f"AIC: {mv['aic']:.1f} | McFadden Pseudo-R²: {mv['pseudo_r2']:.3f} | AUC: {mv['roc_auc']:.3f}")
hl_stat, hl_p = mv['hosmer_lemeshow']
print(f"Hosmer-Lemeshow: χ² = {hl_stat:.2f}, p = {hl_p:.3f}")
if mv['removed_vif']:
    print(f"Removed for VIF > 5: {mv['removed_vif']}")
print()

# Summary table
mv_display = mv['summary'].copy()
mv_display['OR [95% CI]'] = mv_display.apply(
    lambda r: f"{r['or']:.2f} [{r['ci_lower']:.2f} – {r['ci_upper']:.2f}]", axis=1
)
mv_display['p-value'] = mv_display['p_value'].map(lambda x: f"{x:.3f}" if x >= 0.001 else "<0.001")
print("Adjusted Odds Ratios:")
print(mv_display[['predictor', 'OR [95% CI]', 'p-value']].to_string(index=False))
print()

# VIF table
print("Variance Inflation Factors:")
print(mv['vif'].to_string(index=False))

### 11.5 Bootstrap Internal Validation

In [ ]:
# Bootstrap validation (1000 resamples) for the multivariable model
# Uses the same clinically-guided predictors (CAPRA components + time to BF)
from pca_mri.analysis.regression import CANDIDATE_PREDICTORS

_FALLBACK = {"tx-age", "psa-val", "tx-gleason_total", "tx-t_stage",
             "tx-biopsy_positive_ratio", "bf-time_to_bf-days"}
mv_predictors = [(c, l, t) for c, l, t in CANDIDATE_PREDICTORS if c in _FALLBACK]

boot = regression.bootstrap_auc(df, predictor_cols=mv_predictors, n_boot=1000)
print(f"Bootstrap Internal Validation (n_boot={boot['n_successful_boots']})")
print(f"  Apparent AUC:    {boot['apparent_auc']:.3f}")
print(f"  Optimism:        {boot['optimism']:.3f}")
print(f"  Corrected AUC:   {boot['corrected_auc']:.3f}")
print(f"  95% CI:          [{boot['ci_lower']:.3f} – {boot['ci_upper']:.3f}]")

### 11.6 Results Dashboard

Interactive Plotly dashboard combining all analysis results: prevalence, diagnostic accuracy, forest plots, ROC curve, and calibration.

In [ ]:
# Render the full interactive dashboard
panels = show_dashboard(df, run_multivariable=True)